In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

In [7]:
users = pd.read_csv("/content/drive/MyDrive/users_data.csv")
cards = pd.read_csv("/content/drive/MyDrive/cards_data (1).csv")
transactions = pd.read_csv("/content/drive/MyDrive/transactions_data.csv",nrows=300000)

In [8]:
transactions['amount'] = transactions['amount'].replace('[\$,()]','', regex=True)
transactions['amount'] = pd.to_numeric(transactions['amount'], errors='coerce')

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipython-input-2134/487747148.py:1: SyntaxWarning: invalid escape sequence '\$'
  transactions['amount'] = transactions['amount'].replace('[\$,()]','', regex=True)


In [9]:
transactions['amount'].describe()

,amount
count,300000.000000
mean,43.630014
std,84.874196
min,-500.000000
25%,9.030000
50%,29.410000
75%,65.010000
max,4978.450000


In [10]:
merged = transactions.merge(
    users,
    left_on='client_id',
    right_on='id',
    how='left'
)

In [11]:
merged.head()

,id_x,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,...,7,Female,594 Mountain View Street,46.80,-100.76,$23679,$48277,$110153,740,4
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,...,6,Male,604 Pine Street,40.80,-91.12,$18076,$36853,$112139,834,5
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084.0,...,4,Male,2379 Forest Lane,33.18,-117.29,$16894,$34449,$36540,686,3
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,...,5,Female,903 Hill Boulevard,41.42,-87.35,$26168,$53350,$128676,685,5
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776.0,...,5,Male,166 River Drive,38.86,-76.60,$33529,$68362,$96182,711,2


In [12]:
print(merged.columns)
print(cards.columns)

Index(['id_x', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'id_y', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'latitude', 'longitude',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards'],
      dtype='object')
Index(['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web'],
      dtype='object')


In [13]:
merged2 = merged.merge(
    cards,
    left_on='card_id',
    right_on='id',
    how='left'
)

In [14]:
customer_spending = merged.groupby('client_id').agg(
    total_spent = ('amount', 'sum'),
    transaction_count = ('amount', 'count'),
    avg_transaction = ('amount', 'mean')
).reset_index()

In [15]:
customer_spending.head()

,client_id,total_spent,transaction_count,avg_transaction
0,0,15634.57,294,53.178810
1,1,7809.69,235,33.232723
2,2,7898.65,292,27.050171
3,3,4506.86,116,38.852241
4,4,15540.28,368,42.229022


In [21]:
customer_spending = customer_spending.merge(
    users,
    left_on='client_id',
    right_on='id',
    how='left'
)

In [22]:
customer_spending.head()

,client_id,total_spent,transaction_count,avg_transaction,id_x,current_age_x,retirement_age_x,birth_year_x,birth_month_x,gender_x,...,birth_month_y,gender_y,address_y,latitude_y,longitude_y,per_capita_income_y,yearly_income_y,total_debt_y,credit_score_y,num_credit_cards_y
0,0,15634.57,294,53.178810,0,33,69,1986,3,Male,...,3,Male,858 Plum Avenue,43.59,-70.33,$29237,$59613,$36199,763,4
1,1,7809.69,235,33.232723,1,43,74,1976,4,Female,...,4,Female,113 Burns Lane,30.44,-87.18,$22247,$45360,$14587,704,3
2,2,7898.65,292,27.050171,2,48,64,1971,8,Male,...,8,Male,6035 Forest Avenue,40.84,-73.87,$13461,$27447,$80850,673,5
3,3,4506.86,116,38.852241,3,49,65,1970,12,Male,...,12,Male,840 Elm Avenue,33.89,-98.51,$13705,$27943,$18693,681,4
4,4,15540.28,368,42.229022,4,54,72,1965,3,Female,...,3,Female,6016 Little Creek Boulevard,47.61,-122.30,$37485,$76431,$115362,716,5


In [23]:
card_limits = cards.groupby('client_id').agg(
    total_credit_limit = ('credit_limit', 'sum')
).reset_index()

In [19]:
card_limits.head()

,client_id,total_credit_limit
0,0,$17600$31490$31008$25558
1,1,$18105$10900$12800
2,2,$7400$9001$9669$16491$7800
3,3,$80$65$62$13515
4,4,$27520$38648$41051$19300$9683


In [24]:
cards['credit_limit'] = cards['credit_limit'].replace('[\$,()]','', regex=True)
cards['credit_limit'] = pd.to_numeric(cards['credit_limit'], errors='coerce')

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipython-input-2134/546316399.py:1: SyntaxWarning: invalid escape sequence '\$'
  cards['credit_limit'] = cards['credit_limit'].replace('[\$,()]','', regex=True)


In [25]:
cards['credit_limit'].head()

,credit_limit
0,24295
1,21968
2,46414
3,12400
4,28


In [26]:
customer_spending = customer_spending.merge(
    card_limits,
    on='client_id',
    how='left'
)

In [27]:
customer_spending.head()

,client_id,total_spent,transaction_count,avg_transaction,id_x,current_age_x,retirement_age_x,birth_year_x,birth_month_x,gender_x,...,gender_y,address_y,latitude_y,longitude_y,per_capita_income_y,yearly_income_y,total_debt_y,credit_score_y,num_credit_cards_y,total_credit_limit
0,0,15634.57,294,53.178810,0,33,69,1986,3,Male,...,Male,858 Plum Avenue,43.59,-70.33,$29237,$59613,$36199,763,4,105656
1,1,7809.69,235,33.232723,1,43,74,1976,4,Female,...,Female,113 Burns Lane,30.44,-87.18,$22247,$45360,$14587,704,3,41805
2,2,7898.65,292,27.050171,2,48,64,1971,8,Male,...,Male,6035 Forest Avenue,40.84,-73.87,$13461,$27447,$80850,673,5,50361
3,3,4506.86,116,38.852241,3,49,65,1970,12,Male,...,Male,840 Elm Avenue,33.89,-98.51,$13705,$27943,$18693,681,4,13722
4,4,15540.28,368,42.229022,4,54,72,1965,3,Female,...,Female,6016 Little Creek Boulevard,47.61,-122.30,$37485,$76431,$115362,716,5,136202


In [28]:
customer_spending['spending_to_income'] = customer_spending['total_spent'] / customer_spending['yearly_income']

customer_spending['spending_to_credit'] = customer_spending['total_spent'] / customer_spending['total_credit_limit']

KeyError: 'yearly_income'

In [29]:
print(customer_spending.columns)

Index(['client_id', 'total_spent', 'transaction_count', 'avg_transaction',
       'id_x', 'current_age_x', 'retirement_age_x', 'birth_year_x',
       'birth_month_x', 'gender_x', 'address_x', 'latitude_x', 'longitude_x',
       'per_capita_income_x', 'yearly_income_x', 'total_debt_x',
       'credit_score_x', 'num_credit_cards_x', 'id_y', 'current_age_y',
       'retirement_age_y', 'birth_year_y', 'birth_month_y', 'gender_y',
       'address_y', 'latitude_y', 'longitude_y', 'per_capita_income_y',
       'yearly_income_y', 'total_debt_y', 'credit_score_y',
       'num_credit_cards_y', 'total_credit_limit'],
      dtype='object')


In [30]:
customer_spending = customer_spending.rename(columns={
    'yearly_income_x':'yearly_income',
    'credit_score_x':'credit_score'
})

In [31]:
customer_spending = customer_spending.drop(columns=[
    'yearly_income_y',
    'credit_score_y'
])

In [32]:
customer_spending['spending_to_income'] = customer_spending['total_spent'] / customer_spending['yearly_income']

customer_spending['spending_to_credit'] = customer_spending['total_spent'] / customer_spending['total_credit_limit']


TypeError: unsupported operand type(s) for /: 'float' and 'str'

In [33]:
customer_spending['yearly_income'] = customer_spending['yearly_income'].replace('[\$,]','', regex=True)
customer_spending['yearly_income'] = pd.to_numeric(customer_spending['yearly_income'], errors='coerce')

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipython-input-2134/3554382385.py:1: SyntaxWarning: invalid escape sequence '\$'
  customer_spending['yearly_income'] = customer_spending['yearly_income'].replace('[\$,]','', regex=True)


In [34]:
customer_spending['yearly_income'].head()

,yearly_income
0,59613
1,45360
2,27447
3,27943
4,76431


In [35]:
customer_spending['spending_to_income'] = customer_spending['total_spent'] / customer_spending['yearly_income']

customer_spending['spending_to_credit'] = customer_spending['total_spent'] / customer_spending['total_credit_limit']


In [36]:
customer_spending['income_group'] = pd.qcut(
    customer_spending['yearly_income'],
    3,
    labels=['Low Income','Mid Income','High Income']
)

In [37]:
customer_spending['spending_behavior'] = pd.qcut(
    customer_spending['spending_to_income'],
    3,
    labels=['Conservative','Balanced','Aggressive']
)

In [38]:
customer_spending[['income_group','spending_behavior']].head()

,income_group,spending_behavior
0,High Income,Balanced
1,Mid Income,Conservative
2,Low Income,Balanced
3,Low Income,Conservative
4,High Income,Balanced


In [39]:
customer_spending['customer_type'] = (
    customer_spending['income_group'].astype(str)
    + " - " +
    customer_spending['spending_behavior'].astype(str)
)

In [40]:
customer_spending[['income_group','spending_behavior','customer_type']].head()

,income_group,spending_behavior,customer_type
0,High Income,Balanced,High Income - Balanced
1,Mid Income,Conservative,Mid Income - Conservative
2,Low Income,Balanced,Low Income - Balanced
3,Low Income,Conservative,Low Income - Conservative
4,High Income,Balanced,High Income - Balanced


In [41]:
customer_spending.to_csv("customer_financial_profile.csv", index=False)

In [42]:
from google.colab import files
files.download("customer_financial_profile.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>